# Eding STEM TR — QLoRA Fine-tuning

Bu notebook, Türkçe STEM veri setiyle bir dil modelini (`Turkish-Llama-8B`) eğitir. Bütün kod notebook'un içindedir; ayrı dosya kurmana gerek yok.

### Nasıl çalıştırılır
1. Üst menü: **Runtime → Change runtime type → GPU** seç (T4, L4 veya A100).
2. Hücreleri **yukarıdan aşağıya sırayla** çalıştır (her birinde ▶).
3. Kod GPU'nu kendisi tanır ve ayarları otomatik yapar — elle bir şey değiştirmene gerek yok.

Toplam süre GPU'ya göre ~30 dk – 2 saat. Takılırsan ilgili hücrenin üstündeki notu oku.

## 1. GPU'yu kontrol et
Hangi GPU'da olduğunu gösterir. Çıktıda GPU adı ve belleği görünmeli.

In [ ]:
import os
# Bellek parçalanmasını azalt (torch CUDA'yı başlatmadan ÖNCE ayarlanmalı)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
print('CUDA :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU  :', torch.cuda.get_device_name(0))
    cap = torch.cuda.get_device_capability()
    print('Compute capability:', cap, '(>=8.0 -> bf16, degilse fp16; T4=7.5)')
    print('Bellek (GiB):', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
!nvidia-smi -L

## 2. Gerekli kütüphaneleri kur
Sadece iki paket kurulur (`bitsandbytes`, `peft`), birkaç dakika sürer. Sonunda **`bitsandbytes OK`** yazısını görmelisin. Eğer bir hata çıkarsa: üst menü **Runtime → Restart session**, sonra bu hücreden devam et.

In [ ]:
!pip install -q -U bitsandbytes peft
!pip uninstall -y -q torchao
# bitsandbytes gercekten yuklendi mi (ayri islemde test):
!python -c "import bitsandbytes as b; print('bitsandbytes OK ->', b.__version__)"
print('Kurulum bitti. Hata alirsan: Runtime -> Restart session, sonra 2den devam.')

## 3. Proje dosyalarını getir
Model, `data/` klasöründeki veriyle eğitilir. İki yol var, birini seç:

- **GitHub'dan (kolay):** Aşağıda `GITHUB_URL`'yi kendi repo adresinle doldur; hücre otomatik indirir.
- **Google Drive'dan:** Proje zip'ini Drive'a yüklediysen `GITHUB_URL`'yi boş bırak; Drive'dan bulur.

Hangisini seçersen seç, hücre sonunda **train / val / test = 850 / 50 / 100** yazmalı.

In [ ]:
import os, glob, zipfile

GITHUB_URL = ''   # ornek: 'https://github.com/kullanici/eding-stem-tr.git'  (bossa Drive'dan aranir)
NEED = 'data/splits/train.jsonl'

def find_dir(base, maxd=6):
    base=base.rstrip('/'); d0=base.count('/')
    for cur,dirs,_ in os.walk(base):
        if cur.count('/')-d0>maxd: dirs[:]=[]; continue
        if os.path.exists(os.path.join(cur, NEED)): return cur
    return None

def zip_ok(z):
    try:
        with zipfile.ZipFile(z) as f: return any(n.endswith(NEED) for n in f.namelist())
    except Exception: return False

root = None
if os.path.exists(NEED): root = os.getcwd()                 # 1) zaten burada
if root is None and GITHUB_URL:                              # 2) GitHub'dan klonla
    os.system('git clone -q ' + GITHUB_URL + ' /content/repo'); root = find_dir('/content/repo')
if root is None: root = find_dir('/content')                # 3) /content altinda
if root is None:                                            # 4) Google Drive
    from google.colab import drive; drive.mount('/content/drive')
    root = find_dir('/content/drive/MyDrive')
    if root is None:
        zips = [z for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True) if zip_ok(z)]
        assert zips, 'Veri bulunamadi. GITHUB_URL doldur ya da proje zip ini Drive a yukle.'
        print('Zip:', zips[0])
        with zipfile.ZipFile(zips[0]) as f: f.extractall('/content/_proj')
        root = find_dir('/content/_proj')

assert root, 'data/splits bulunamadi.'
os.chdir(root)                       # ciktilar (outputs/) bu klasorde olusur
DATA_DIR = os.path.join(root, 'data', 'splits')
print('Proje klasoru:', root)
print('train:', sum(1 for _ in open(os.path.join(DATA_DIR,'train.jsonl'))),
      '| val:', sum(1 for _ in open(os.path.join(DATA_DIR,'validation.jsonl'))),
      '| test:', sum(1 for _ in open(os.path.join(DATA_DIR,'test.jsonl'))))

## 4. Eğitim kodunu yükle
Bu hücre eğitim fonksiyonlarını **tanımlar** (henüz bir şey eğitmez). Çıktı olarak `Egitim kodu yuklendi` yazması yeterli.

Bilmen gerekenler (elle değiştirmene gerek yok, hepsi otomatik):
- **Ayarlar GPU'ya göre otomatik seçilir** (küçük GPU'da küçük batch, büyük GPU'da büyük).
- **Kalite ayarları açık:** model tüm katmanlara uygulanır + NEFTune + en iyi kopyayı seçme.
- İnternet/indirme hatası (429) olursa model yüklemeyi kendi tekrar dener.

İleri seviye: aşağıdaki `CONFIG` içinden epoch sayısı gibi ayarları değiştirebilirsin.

In [ ]:
import os, json, time, torch
from datasets import Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

CONFIG = {
    "base_model": "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1",
    "output_dir": "outputs/qlora-checkpoints",
    "lora": {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
             "target_modules": ["q_proj", "v_proj"], "bias": "none"},
    "quant": {"load_in_4bit": True, "bnb_4bit_quant_type": "nf4",
              "bnb_4bit_use_double_quant": True},
    # T4 (16GB) icin guvenli: batch 1 x birikim 16 = efektif 16, seq 512.
    "args": {"num_train_epochs": 3, "per_device_train_batch_size": 1,
             "gradient_accumulation_steps": 16, "learning_rate": 2e-4,
             "weight_decay": 0.01, "warmup_ratio": 0.03, "lr_scheduler_type": "cosine",
             "max_seq_length": 512, "logging_steps": 10, "save_steps": 50,
             "eval_steps": 50, "gradient_checkpointing": True,
             "optim": "paged_adamw_32bit", "seed": 42},
}

# --- Kalite ayari ---
# True  = LoRA TUM linear katmanlara (q,k,v,o,gate,up,down) -> EN IYI KALITE (QLoRA makalesi).
#         Egitilebilir ~41.9M (yine %99.5 azaltma).
# False = yalnizca q_proj/v_proj (~6.8M, %99.9 azaltma; minimal).
QUALITY_ALL_LINEAR = True
ALL_LINEAR = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
NEFTUNE_ALPHA = 5   # NEFTune gurultu (kalite artisi); 0 -> kapali

def auto_tune():
    """GPU belleğine göre en verimli batch/seq'i seçer (efektif batch = 16 sabit)."""
    if not torch.cuda.is_available():
        return
    gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    a = CONFIG["args"]
    if gb >= 38:      # A100 40GB
        a.update(per_device_train_batch_size=8, gradient_accumulation_steps=2, max_seq_length=1024)
    elif gb >= 22:    # L4 24GB
        a.update(per_device_train_batch_size=4, gradient_accumulation_steps=4, max_seq_length=1024)
    else:             # T4 16GB
        a.update(per_device_train_batch_size=1, gradient_accumulation_steps=16, max_seq_length=512)
    eff = a["per_device_train_batch_size"] * a["gradient_accumulation_steps"]
    print(f"[AUTO] GPU ~{gb:.0f}GB -> batch {a['per_device_train_batch_size']} x "
          f"accum {a['gradient_accumulation_steps']} (efektif {eff}), seq {a['max_seq_length']}, "
          f"precision {'bf16' if bf16_ok() else 'fp16'}")

CHAT_TEMPLATE = (
"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
"Sen bir Turkce K-12 STEM ve kodlama egitimi asistanisin. Ogrencilere Arduino, "
"Scratch, mBlock, robotik, Python ve elektronik konularinda yardim ediyorsun. "
"Cevaplarini Turkce ver, kod iceren cevaplarda her satiri acikla."
"<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
"{instruction}{input_section}<|eot_id|>"
"<|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>")
RESPONSE_MARKER = "<|start_header_id|>assistant<|end_header_id|>\n\n"

def bf16_ok():
    # bf16 yalnizca Ampere+ (compute capability >= 8.0) mimaride native.
    # T4 = 7.5 -> fp16 (daha hizli ve az bellek).
    if not torch.cuda.is_available():
        return False
    try:
        return torch.cuda.get_device_capability()[0] >= 8
    except Exception:
        return torch.cuda.is_bf16_supported()
def dtype():
    return torch.bfloat16 if bf16_ok() else torch.float16

def read_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

def format_example(ex):
    inp = f"\n\nEk bilgi: {ex['input']}" if ex.get("input", "").strip() else ""
    return CHAT_TEMPLATE.format(instruction=ex["instruction"], input_section=inp, output=ex["output"])

def build_dataset(rows, tok, max_len):
    out = []
    for ex in rows:
        full = format_example(ex)
        cut = full.find(RESPONSE_MARKER)
        prompt = full[:cut + len(RESPONSE_MARKER)]
        full_ids = tok(full, add_special_tokens=False, truncation=True, max_length=max_len)["input_ids"]
        prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        n = min(len(prompt_ids), len(full_ids))
        labels = [-100]*n + full_ids[n:]
        if any(l != -100 for l in labels):
            out.append({"input_ids": full_ids, "attention_mask": [1]*len(full_ids), "labels": labels})
    return Dataset.from_list(out)

def bnb_config():
    q = CONFIG["quant"]
    print("[INFO] Compute dtype:", "bfloat16" if bf16_ok() else "float16 (T4)")
    return BitsAndBytesConfig(load_in_4bit=q["load_in_4bit"], bnb_4bit_quant_type=q["bnb_4bit_quant_type"],
                              bnb_4bit_compute_dtype=dtype(), bnb_4bit_use_double_quant=q["bnb_4bit_use_double_quant"])

def is_transient(e):
    s = str(e).lower()
    return any(k in s for k in ["429","too many requests","rate limit","timeout","timed out",
                                "connection","temporarily","max retries","queue size","503","502"])

def load_model_and_tokenizer():
    name = CONFIG["base_model"]
    print("\n[INFO] Model yukleniyor:", name)
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token; tok.pad_token_id = tok.eos_token_id
    tok.padding_side = "right"
    try:
        import bitsandbytes  # noqa
        bnb_ok = True
    except Exception as e:
        bnb_ok = False; print("[WARN] bitsandbytes yok:", e)
    model = None; quantized = False
    if bnb_ok:
        cfg = bnb_config()
        for attempt in range(4):
            try:
                model = AutoModelForCausalLM.from_pretrained(name, quantization_config=cfg,
                                                             device_map="auto", trust_remote_code=True)
                quantized = True; print("[INFO] 4-bit (QLoRA) yukleme basarili."); break
            except Exception as e:
                if is_transient(e) and attempt < 3:
                    w = 30*(attempt+1)
                    print(f"[WARN] Gecici hata (deneme {attempt+1}/4), {w}s bekleniyor: {str(e)[:140]}")
                    time.sleep(w); continue
                if is_transient(e):
                    raise RuntimeError("HF gecici hata (429). Model onbellege alindi; bu hucreyi TEKRAR calistir.") from e
                print("[WARN] 4-bit yuklenemedi:", str(e)[:200]); break
    if model is None:
        print("[WARN] bf16/fp16 LoRA'ya geciliyor (8B ~16GB; kucuk GPU'da yavas).")
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype(),
                                                     device_map="auto", trust_remote_code=True)
    model.config.use_cache = False
    if quantized:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        if hasattr(model, "enable_input_require_grads"): model.enable_input_require_grads()
    print(f"[INFO] Toplam parametre: {sum(p.numel() for p in model.parameters()):,} | "
          f"Kuantizasyon: {'4-bit NF4' if quantized else 'yok'}")
    return model, tok, quantized

def add_lora(model):
    lc = CONFIG["lora"]
    targets = ALL_LINEAR if QUALITY_ALL_LINEAR else lc["target_modules"]
    print("[INFO] LoRA hedef moduller:", targets)
    model = get_peft_model(model, LoraConfig(r=lc["r"], lora_alpha=lc["lora_alpha"],
        lora_dropout=lc["lora_dropout"], target_modules=targets,
        bias=lc["bias"], task_type=TaskType.CAUSAL_LM))
    model.print_trainable_parameters()
    return model

def training_args(quantized):
    a = CONFIG["args"]
    optim = a["optim"] if quantized else "adamw_torch"
    kw = dict(output_dir=CONFIG["output_dir"], num_train_epochs=a["num_train_epochs"],
        per_device_train_batch_size=a["per_device_train_batch_size"],
        per_device_eval_batch_size=a["per_device_train_batch_size"],
        gradient_accumulation_steps=a["gradient_accumulation_steps"],
        learning_rate=a["learning_rate"], weight_decay=a["weight_decay"],
        warmup_ratio=a["warmup_ratio"], lr_scheduler_type=a["lr_scheduler_type"],
        logging_steps=a["logging_steps"], save_steps=a["save_steps"], save_strategy="steps",
        fp16=not bf16_ok(), bf16=bf16_ok(), gradient_checkpointing=a["gradient_checkpointing"],
        gradient_checkpointing_kwargs={"use_reentrant": False}, optim=optim,
        seed=a["seed"], report_to="none", save_total_limit=2)
    fields = getattr(TrainingArguments, "__dataclass_fields__", {})
    kw["eval_strategy" if "eval_strategy" in fields else "evaluation_strategy"] = "steps"
    kw["eval_steps"] = a["eval_steps"]
    if "load_best_model_at_end" in fields:
        kw.update(load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False)
    if NEFTUNE_ALPHA and "neftune_noise_alpha" in fields:
        kw["neftune_noise_alpha"] = NEFTUNE_ALPHA   # kalite artisi
    kw = {k: v for k, v in kw.items() if (not fields) or k in fields}
    return TrainingArguments(**kw)

def train(data_dir):
    print("="*60 + "\nQLoRA Fine-tuning (self-contained)\n" + "="*60)
    auto_tune()   # GPU'ya göre batch/seq'i otomatik ayarla
    train_rows = read_jsonl(os.path.join(data_dir, "train.jsonl"))
    val_rows = read_jsonl(os.path.join(data_dir, "validation.jsonl"))
    print(f"[INFO] train {len(train_rows)} | val {len(val_rows)}")
    model, tok, quantized = load_model_and_tokenizer()
    model = add_lora(model)
    ml = CONFIG["args"]["max_seq_length"]
    tr = build_dataset(train_rows, tok, ml)
    ev = build_dataset(val_rows, tok, ml)
    print(f"[INFO] tokenize -> train {len(tr)} | val {len(ev)}")
    trainer = Trainer(model=model, args=training_args(quantized), train_dataset=tr, eval_dataset=ev,
                      data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100, return_tensors="pt"))
    print("\n[INFO] Egitim basliyor...")
    res = trainer.train()
    final = os.path.join(CONFIG["output_dir"], "final-adapter")
    trainer.save_model(final); tok.save_pretrained(final)
    with open(os.path.join(CONFIG["output_dir"], "training_metrics.json"), "w") as f:
        json.dump(res.metrics, f, indent=2)
    print("\n[OK] Adapter kaydedildi:", final)
    return trainer

def test_inference(adapter, prompts=None):
    from peft import PeftModel
    prompts = prompts or ["Arduino ile servo motor nasil kontrol edilir?",
                          "Python'da 1'den 100'e kadar toplami bulan program yaz."]
    name = CONFIG["base_model"]
    try:
        base = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb_config(), device_map="auto")
    except Exception:
        base = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype(), device_map="auto")
    model = PeftModel.from_pretrained(base, adapter); tok = AutoTokenizer.from_pretrained(adapter); model.eval()
    for p in prompts:
        t = CHAT_TEMPLATE.format(instruction=p, input_section="", output="").split(RESPONSE_MARKER)[0] + RESPONSE_MARKER
        ins = tok(t, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**ins, max_new_tokens=400, do_sample=True, temperature=0.7,
                                 top_p=0.9, repetition_penalty=1.15, eos_token_id=tok.eos_token_id)
        print(f"\n[SORU] {p}\n[CEVAP] {tok.decode(out[0][ins['input_ids'].shape[1]:], skip_special_tokens=True)[:500]}\n"+"-"*40)

print("Egitim kodu yuklendi. (train, test_inference hazir)")

## 5. Eğitimi başlat
Bu hücre eğitimi çalıştırır. Yaklaşık süre (3 epoch): **A100 ~20–30 dk · L4 ~40–60 dk · T4 ~1.5–2 saat**. Bittiğinde `EGITIM TAMAMLANDI` yazar.

> **429** (indirme limiti) çıkarsa bu hücreyi bir kez daha çalıştır — model önbellekten yüklenir.
> **OOM** (bellek doldu) çıkarsa 4. hücrede `CONFIG['args']['max_seq_length']`'i 384 yap, 4–5'i tekrar çalıştır.
> **Eğitim biter bitmez 6. hücreyi çalıştır (modeli kaydet), yoksa oturum kapanınca silinir.**

In [ ]:
trainer = train(DATA_DIR)
print('\nEGITIM TAMAMLANDI. Simdi 6. hucre: modeli kaydet.')

## 6. Modeli Drive'a kaydet — ÖNEMLİ
Colab oturumu kapandığında eğittiğin model **silinir**. Bu hücre modeli Google Drive'a kopyalar. Küçük dosyadır, birkaç saniye sürer. Eğitim biter bitmez çalıştır.

In [ ]:
import shutil
from google.colab import drive; drive.mount('/content/drive')
DST = '/content/drive/MyDrive/eding-stem-adapter'
shutil.copytree('outputs/qlora-checkpoints/final-adapter', DST, dirs_exist_ok=True)
print('Model Drive a kaydedildi:', DST)

## 7. Hızlı test
Modele birkaç örnek soru sorar. Cevaplar Türkçe ve kısa olmalı — modelin öğrendiğini buradan görürsün.

In [ ]:
test_inference('outputs/qlora-checkpoints/final-adapter')

## 8. Değerlendirme — model base'ten iyi mi?
Test setinde hem **eğitilmiş** hem **eğitilmemiş (base)** modeli çalıştırıp BLEU / ROUGE-L / BERTScore hesaplar. ~20–40 dk sürer. Sonuçlar `outputs/evaluation/results.json`'a yazılır.

Beklenti: eğitilmiş modelin sayıları base'ten belirgin yüksek olmalı (asıl kanıtın budur).

In [ ]:
!pip install -q sacrebleu rouge-score bert-score
import gc, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

for _n in ['trainer','model','ft','bm','base']:
    if _n in globals():
        try: del globals()[_n]
        except: pass
gc.collect(); torch.cuda.empty_cache()

NAME = CONFIG["base_model"]; ADAPTER = "outputs/qlora-checkpoints/final-adapter"
test_rows = read_jsonl(os.path.join(DATA_DIR, "test.jsonl"))
refs = [r["output"] for r in test_rows]
tok = AutoTokenizer.from_pretrained(ADAPTER)
_eot = tok.convert_tokens_to_ids("<|eot_id|>")
EOS = list({tok.eos_token_id, _eot})

def _prompt(instr):
    t = CHAT_TEMPLATE.format(instruction=instr, input_section="", output="")
    return t.split(RESPONSE_MARKER)[0] + RESPONSE_MARKER

@torch.no_grad()
def gen_all(model):
    out = []
    for i, r in enumerate(test_rows):
        ins = tok(_prompt(r["instruction"]), return_tensors="pt").to(model.device)
        o = model.generate(**ins, max_new_tokens=400, do_sample=False,
                           repetition_penalty=1.15, eos_token_id=EOS, pad_token_id=tok.eos_token_id)
        out.append(tok.decode(o[0][ins["input_ids"].shape[1]:], skip_special_tokens=True).strip())
        if (i+1) % 20 == 0: print("  ", i+1, "/", len(test_rows))
    return out

def _load(adapter=False):
    try: m = AutoModelForCausalLM.from_pretrained(NAME, quantization_config=bnb_config(), device_map="auto")
    except Exception: m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype(), device_map="auto")
    if adapter: m = PeftModel.from_pretrained(m, ADAPTER)
    m.eval(); return m

print("Fine-tuned uretiyor..."); ft = _load(adapter=True); ft_p = gen_all(ft)
del ft; gc.collect(); torch.cuda.empty_cache()
print("Base uretiyor...");       bm = _load(adapter=False); bl_p = gen_all(bm)
del bm; gc.collect(); torch.cuda.empty_cache()

import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bscore
_rs = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
def metrics(preds):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    rl = 100*sum(_rs.score(r,p)["rougeL"].fmeasure for r,p in zip(refs,preds))/len(refs)
    _,_,F = bscore(preds, refs, lang="tr", verbose=False)
    return {"BLEU": round(bleu,2), "ROUGE-L": round(rl,2), "BERTScore-F1": round(100*F.mean().item(),2)}

res = {"fine_tuned": metrics(ft_p), "baseline": metrics(bl_p)}
print("\n=== SONUC ==="); print(json.dumps(res, ensure_ascii=False, indent=2))
os.makedirs("outputs/evaluation", exist_ok=True)
json.dump({"metrics": res, "ornekler": [
    {"soru": test_rows[i]["instruction"], "base": bl_p[i], "fine_tuned": ft_p[i], "referans": refs[i]}
    for i in range(min(5, len(test_rows)))]},
    open("outputs/evaluation/results.json","w"), ensure_ascii=False, indent=2)
print("\nKaydedildi -> outputs/evaluation/results.json")

## 9. HuggingFace'e yükle — opsiyonel
Modeli herkese açık paylaşmak istersen. Önce huggingface.co → Settings → Access Tokens → **Write** yetkili token oluştur. Hücre çalışınca çıkan **kullanıcı adını** `REPO` satırına yaz.

In [ ]:
from huggingface_hub import login, whoami
login()                                   # Write yetkili token'i yapistir
print("HF kullanici adin:", whoami()["name"])   # asagidaki REPO'da bunu kullan

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

REPO    = "KULLANICI/Turkish-Llama-8B-STEM-QLoRA"   # <-- KULLANICI'yi yukaridaki adinla degistir
ADAPTER = "outputs/qlora-checkpoints/final-adapter" # (yeni oturumda: Drive yolunu ver)

base = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], torch_dtype=torch.float16, device_map="auto")
m = PeftModel.from_pretrained(base, ADAPTER)
m.push_to_hub(REPO)
AutoTokenizer.from_pretrained(ADAPTER).push_to_hub(REPO)
print("Yuklendi -> https://huggingface.co/" + REPO)